# 04 - Single-line-to-ground fault

## Objective

Apply a declared single-line-to-ground fault and compare the solver-returned current from direct OpenDSS with the CEPT public CLI. Distinguish a calculated current from a protection-duty decision.

## Source, assumptions, and units

The source is the IEEE13 feeder bundled in the installed CEPT wheel. The fault is an SLG fault on phase 1 of bus `675` with `rf = 0.001 ohm`. Current is A, fault resistance is ohm, and voltage observations are pu. The source and fault settings are demonstrator assumptions, not field data or certified protection inputs.

## Prediction

The fault-element current magnitude should be positive and finite. Direct OpenDSS and CEPT should agree within the declared teaching tolerance because the fault type, bus, phase, resistance, and feeder source are matched.

## Action

Solve the same fault directly, then stream `cept study demo fault` into an exact run directory.

## Verification

Read phase currents from CEPT's persisted `results.json`, run `cept study verify`, and compare the total solver-returned current.

## Interpretation

The current depends on the declared source and feeder model. A public workflow receipt does not certify interrupting duty, relay settings, arc-flash analysis, or a field short-circuit result.

## Exercise

Change one explicit fault input in the direct command and in the lesson's declared prediction, such as `FAULT_RESISTANCE_OHM`. Explain how that change should affect current, then restart and rerun all cells.

## Runtime requirements

Use Python 3.10 or newer with an existing installed `cept` command, or provide a caller-owned wheel through `CEPT_WHEEL_URL` and its exact `CEPT_WHEEL_SHA256`. The wheel must provide CEPT, OpenDSSDirect.py, and the bundled IEEE13 source files. No released PyPI version is assumed. Jupyter is needed only to execute the notebook.

In [ ]:
import hashlib
import importlib.util
import json
import os
import shlex
import shutil
import subprocess
import sys
import urllib.parse
import urllib.request
from importlib.resources import files
from pathlib import Path

DEFAULT_WHEEL_URL = 'https://github.com/sarutesri/cept-studio-edu/releases/download/v0.2.0-edu.1/cept_power_studio-0.2.0.dev0-py3-none-any.whl'
DEFAULT_WHEEL_SHA256 = 'c7e609a1d9cc85b322bfb615f0c796c7ea5c43815b197289eb555786964478bc'
configured_url = os.environ.get('CEPT_WHEEL_URL')
WHEEL_URL = (DEFAULT_WHEEL_URL if configured_url is None and importlib.util.find_spec('cept') is None else (configured_url or '')).strip()
WHEEL_SHA256 = os.environ.get('CEPT_WHEEL_SHA256', DEFAULT_WHEEL_SHA256 if WHEEL_URL == DEFAULT_WHEEL_URL else '').strip().lower()
if WHEEL_URL:
    if len(WHEEL_SHA256) != 64 or any(character not in '0123456789abcdef' for character in WHEEL_SHA256):
        raise ValueError('CEPT_WHEEL_SHA256 must be the caller-provided 64-character SHA-256')
    wheel_path = Path.cwd() / Path(urllib.parse.urlparse(WHEEL_URL).path).name
    print(f'Downloading caller-provided wheel: {WHEEL_URL}')
    urllib.request.urlretrieve(WHEEL_URL, wheel_path)
    digest = hashlib.sha256(wheel_path.read_bytes()).hexdigest()
    if digest != WHEEL_SHA256:
        raise ValueError(f'wheel hash mismatch: expected {WHEEL_SHA256}, got {digest}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', str(wheel_path)], check=True)
else:
    print('CEPT_WHEEL_URL not supplied; using the existing installed environment.')
CLI = [sys.executable, '-m', 'cept.public_cli']
print('CLI:', shlex.join([*CLI, '--version']))
print(subprocess.run([*CLI, '--version'], capture_output=True, text=True, check=True).stdout.strip())
def run_cli(*arguments):
    command = [*CLI, *[str(argument) for argument in arguments]]
    print('$ ' + shlex.join(command), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=Path.cwd())
    lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    returncode = process.wait()
    output = ''.join(lines)
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, command, output=output)
    return json.loads(output) if output.strip().startswith('{') else output

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def show_table(headers, rows):
    print('| ' + ' | '.join(headers) + ' |')
    print('| ' + ' | '.join('---' for _ in headers) + ' |')
    for row in rows:
        print('| ' + ' | '.join(str(value) for value in row) + ' |')

MASTER_DSS = Path(str(files('cept').joinpath('testsystems', 'ieee13', 'IEEE13Nodeckt.dss')))
FAULT_BUS = '675'
FAULT_PHASE = 1
FAULT_RESISTANCE_OHM = 0.001


CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: '<installed-python>' -m cept.public_cli --version


cept-power-studio 0.2.0.dev0


In [ ]:
import opendssdirect as dss

dss.Basic.ClearAll()
dss.Basic.DataPath(str(MASTER_DSS.parent))
dss.Text.Command(f'Redirect \"{MASTER_DSS}\"')
dss.Text.Command(f'New Fault.lesson_fault Bus1={FAULT_BUS}.{FAULT_PHASE} phases=1 r={FAULT_RESISTANCE_OHM}')
dss.Text.Command('Solve')
assert dss.Solution.Converged()
dss.Circuit.SetActiveElement('Fault.lesson_fault')
currents = dss.CktElement.Currents()
direct_current_a = abs(complex(currents[0], currents[1]))
show_table(['source', 'bus', 'phase', 'fault resistance', 'current', 'units'], [('direct OpenDSS', FAULT_BUS, FAULT_PHASE, FAULT_RESISTANCE_OHM, direct_current_a, 'ohm / A')])
assert direct_current_a > 0


| source | bus | phase | fault resistance | current | units |
| --- | --- | --- | --- | --- | --- |
| direct OpenDSS | 675 | 1 | 0.001 | 2950.698148621237 | ohm / A |


In [ ]:
RUN_DIR = Path.cwd() / 'runs' / '04-fault-study'
run_summary = run_cli('study', 'demo', 'fault', '--network', 'ieee13', '--out', RUN_DIR, '--force')
verify_summary = run_cli('study', 'verify', RUN_DIR)
results = read_json(RUN_DIR / 'results.json')
fault = results['fault']
show_table(['source', 'bus', 'fault type', 'phase', 'fault resistance', 'current', 'units'], [('CEPT results.json', fault['bus'], fault['fault_type'], phase['phase'], fault['rf_ohm'], phase['i_amp'], 'ohm / A') for phase in fault['currents']])
cept_current_a = float(fault['total_fault_current_a'])
show_table(['source', 'total fault current', 'unit'], [('direct OpenDSS', direct_current_a, 'A'), ('CEPT results.json', cept_current_a, 'A')])
assert run_summary['status'] == 'PASS'
assert verify_summary['passed'] is True
assert fault['bus'].lower() == FAULT_BUS.lower() and fault['fault_type'] == 'slg'
assert fault['phases'] == [FAULT_PHASE]
assert abs(direct_current_a - cept_current_a) < 1.0


$ '<installed-python>' -m cept.public_cli study demo fault --network ieee13 --out '<installed-cept>\testsystems\ieee13\runs\04-fault-study' --force


{


  "status": "PASS",


  "claim": "WORKFLOW_VALIDATED",


  "study_type": "fault",


...

  "schema": "cept-public-verification-v1",


  "status": "PASS",


  "study_type": "fault"


}


| source | bus | fault type | phase | fault resistance | current | units |
| --- | --- | --- | --- | --- | --- | --- |
| CEPT results.json | 675 | slg | 1 | 0.001 | 2950.7 | ohm / A |
| source | total fault current | unit |
| --- | --- | --- |
| direct OpenDSS | 2950.698148621237 | A |
| CEPT results.json | 2950.7 | A |


The displayed current is read from each solver route, and CEPT verification is performed from the exact persisted run. The result is a bounded `WORKFLOW_VALIDATED` demonstrator observation, not protection or project acceptance.